In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import os
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


#load df_clean and convert to pandas df
df = spark.read.table("novasend_fulfillment.processed.features_engineered").toPandas()

print(f"Shape: {df.shape}")
print(f"Nulls: {df.isnull().sum().sum()}")
print(df.dtypes)

In [0]:
# Load the engineered feature table from Delta catalog
df = spark.read.table("novasend_fulfillment.processed.features_engineered").toPandas()

# Separate features from target
X = df.drop(columns=["Late_delivery_risk"])
y = df["Late_delivery_risk"]

# 80/20 stratified split and preserve the 55/45 class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

In [0]:
# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames so SFS can reference column names
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print(f"Scaled train shape: {X_train_scaled_df.shape}")
print(f"Scaled test shape: {X_test_scaled_df.shape}")

In [0]:
# Stepwise feature selection
logr = LogisticRegression(max_iter=1000, solver="newton-cg", C=np.inf)

sfs = SFS(
    logr,
    k_features="best",
    forward=True,
    floating=True,
    scoring="roc_auc",
    cv=5,
    verbose=2
)

# Fit on scaled training data only
sfs = sfs.fit(X_train_scaled_df, y_train)

# Extract the names of selected features
selected_features = list(sfs.k_feature_names_)

print(f"Number of features selected: {len(selected_features)}")
print(f"Selected features: {selected_features}")

In [0]:
# Store selected features as a constant
SELECTED_FEATURES = selected_features

# Scaled versions for logistic regression
X_train_lr = X_train_scaled_df[SELECTED_FEATURES]
X_test_lr = X_test_scaled_df[SELECTED_FEATURES]

# Unscaled versions for XGBoost and LightGBM
X_train_tree = X_train[SELECTED_FEATURES]
X_test_tree = X_test[SELECTED_FEATURES]

print(f"Logistic regression input shape: {X_train_lr.shape}")
print(f"Tree model input shape: {X_train_tree.shape}")

In [0]:
# Start an MLflow run for logistic regression baseline
with mlflow.start_run(run_name="logistic_regression_baseline"):

    # Same logistic regression config used in stepwise selection
    # C=np.inf disables regularization
    lr_model = LogisticRegression(max_iter=1000, solver="newton-cg", C=np.inf)

    # Train on scaled reduced feature set
    lr_model.fit(X_train_lr, y_train)

    # Generate predictions on scaled test set
    y_pred_lr = lr_model.predict(X_test_lr)
    y_prob_lr = lr_model.predict_proba(X_test_lr)[:, 1]

    # Calculate evaluation metrics
    auc_lr = roc_auc_score(y_test, y_prob_lr)
    f1_lr = f1_score(y_test, y_pred_lr)

    # Log model config to MLflow
    mlflow.log_param("model", "logistic_regression")
    mlflow.log_param("n_features", len(SELECTED_FEATURES))
    mlflow.log_param("solver", "newton-cg")
    mlflow.log_param("C", np.inf)

    # Log metrics to MLflow
    mlflow.log_metric("auc", auc_lr)
    mlflow.log_metric("f1", f1_lr)

    # Log the model artifact
    mlflow.sklearn.log_model(
        lr_model,
        "logistic_regression_model",
        input_example=X_train_lr.iloc[:5]  # fixes the missing signature warning
    )

    print(f"Logistic Regression AUC: {auc_lr:.4f}")
    print(f"Logistic Regression F1:  {f1_lr:.4f}")

In [0]:
# Start an MLflow run for XGBoost on the stepwise-selected feature set
with mlflow.start_run(run_name="xgboost_reduced"):

    # XGBoost with same baseline params as before for fair comparison
    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )

    # Train on unscaled reduced feature set
    xgb_model.fit(X_train_tree, y_train)

    # Generate predictions on unscaled test set
    y_pred_xgb = xgb_model.predict(X_test_tree)
    y_prob_xgb = xgb_model.predict_proba(X_test_tree)[:, 1]

    # Calculate evaluation metrics
    auc_xgb = roc_auc_score(y_test, y_prob_xgb)
    f1_xgb = f1_score(y_test, y_pred_xgb)

    # Log model config to MLflow
    mlflow.log_param("model", "xgboost_reduced")
    mlflow.log_param("n_features", len(SELECTED_FEATURES))
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)

    # Log metrics to MLflow
    mlflow.log_metric("auc", auc_xgb)
    mlflow.log_metric("f1", f1_xgb)

    # Log model artifact
    mlflow.sklearn.log_model(
        xgb_model,
        name="xgboost_reduced_model",
        input_example=X_train_tree.iloc[:5]
    )

    print(f"XGBoost Reduced AUC: {auc_xgb:.4f}")
    print(f"XGBoost Reduced F1:  {f1_xgb:.4f}")

In [0]:
# Start an MLflow run for LightGBM on the stepwise-selected feature set
with mlflow.start_run(run_name="lgbm_reduced"):

    # LightGBM with same baseline params as before for fair comparison
    lgbm_model = LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    # Train on unscaled reduced feature set
    lgbm_model.fit(X_train_tree, y_train)

    # Generate predictions on unscaled test set
    y_pred_lgbm = lgbm_model.predict(X_test_tree)
    y_prob_lgbm = lgbm_model.predict_proba(X_test_tree)[:, 1]

    # Calculate evaluation metrics
    auc_lgbm = roc_auc_score(y_test, y_prob_lgbm)
    f1_lgbm = f1_score(y_test, y_pred_lgbm)

    # Log model config to MLflow
    mlflow.log_param("model", "lgbm_reduced")
    mlflow.log_param("n_features", len(SELECTED_FEATURES))
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)

    # Log metrics to MLflow
    mlflow.log_metric("auc", auc_lgbm)
    mlflow.log_metric("f1", f1_lgbm)

    # Log model artifact
    mlflow.sklearn.log_model(
        lgbm_model,
        name="lgbm_reduced_model",
        input_example=X_train_tree.iloc[:5]
    )

    print(f"LightGBM Reduced AUC: {auc_lgbm:.4f}")
    print(f"LightGBM Reduced F1:  {f1_lgbm:.4f}")

In [0]:
# connect to the local Databricks workspace
mlflow.set_tracking_uri("databricks")

# Set the experiment.
mlflow.set_experiment("/novasend-fulfillment-risk")

In [0]:
# Define feature columns
X = df.drop(columns=["Late_delivery_risk"])

# Define the target variable
y = df["Late_delivery_risk"]

# Split into train and test sets: 80/20 split, stratified to preserve the 45/55 class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Confirm shapes look right
print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

In [0]:
# Start an MLflow run
with mlflow.start_run(run_name="xgboost_baseline"):

    # Define the model
    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42
    )

    # Train the model
    xgb_model.fit(X_train, y_train)

    # Generate predictions and probability scores on the test set
    y_pred = xgb_model.predict(X_test)
    y_prob = xgb_model.predict_proba(X_test)[:, 1]

    # Calculate evaluation metrics
    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)

    # Log hyperparameters to MLflow
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)

    # Log metrics to MLflow
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("f1", f1)

    # Log the trained model to Databricks MLflow model registry
    mlflow.sklearn.log_model(xgb_model, "xgboost_model")

    print(f"XGBoost AUC: {auc:.4f}")
    print(f"XGBoost F1: {f1:.4f}")

In [0]:
# Start a new MLflow run for LightGBM
with mlflow.start_run(run_name="lightgbm_baseline"):

    # Define LightGBM model
    lgbm_model = LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    # Train the model
    lgbm_model.fit(X_train, y_train)

    # Generate predictions and probability scores on the test set
    y_pred = lgbm_model.predict(X_test)
    y_prob = lgbm_model.predict_proba(X_test)[:, 1]

    # Calculate evaluation metrics
    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)

    # Log hyperparameters to MLflow
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)

    # Log metrics to MLflow
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("f1", f1)

    # Add input example so MLflow can auto-infer the model signature
    input_example = X_train.iloc[:5]

    # Log the trained model with signature to satisfy Databricks UC requirements
    mlflow.sklearn.log_model(
        lgbm_model,
        name="lightgbm_model",
        input_example=input_example
    )

    print(f"LightGBM AUC: {auc:.4f}")
    print(f"LightGBM F1: {f1:.4f}")

In [0]:
from sklearn.model_selection import RandomizedSearchCV

# Define parameter space for XGBoost
xgb_param_grid = {
    "n_estimators": [300, 500, 700],
    "max_depth": [4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9]
}

# Define parameter space for LightGBM
lgbm_param_grid = {
    "n_estimators": [300, 500, 700],
    "max_depth": [4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9]
}

# Initialize base models without any hyperparameters
xgb_base = XGBClassifier(eval_metric="logloss", random_state=42)
lgbm_base = LGBMClassifier(random_state=42)

# Run RandomizedSearchCV for XGBoost
xgb_search = RandomizedSearchCV(
    xgb_base,
    xgb_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1,  # use all available CPU cores
    verbose=1
)

# Run RandomizedSearchCV for LightGBM
lgbm_search = RandomizedSearchCV(
    lgbm_base,
    lgbm_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Running XGBoost search...")
xgb_search.fit(X_train, y_train)
print(f"Best XGBoost params: {xgb_search.best_params_}")
print(f"Best XGBoost CV AUC: {xgb_search.best_score_:.4f}")

print("\nRunning LightGBM search...")
lgbm_search.fit(X_train, y_train)
print(f"Best LightGBM params: {lgbm_search.best_params_}")
print(f"Best LightGBM CV AUC: {lgbm_search.best_score_:.4f}")

In [0]:
from sklearn.model_selection import RandomizedSearchCV

# Parameter search space for XGBoost
# These ranges are informed by the baseline run results
xgb_param_grid = {
    "n_estimators": [300, 500, 700],
    "max_depth": [4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9]
}

# Base XGBoost model
xgb_base = XGBClassifier(eval_metric="logloss", random_state=42)

# n_iter=20 tests 20 random combinations
# cv=5 is consistent with our feature selection fold count
# n_jobs=-1 uses all available cores
xgb_search = RandomizedSearchCV(
    xgb_base,
    xgb_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Running XGBoost tuning on reduced feature set...")
xgb_search.fit(X_train_tree, y_train)

print(f"Best XGBoost params: {xgb_search.best_params_}")
print(f"Best XGBoost CV AUC: {xgb_search.best_score_:.4f}")

In [0]:
with mlflow.start_run(run_name="xgboost_tuned"):

    # Build final model using the best params found by RandomizedSearchCV
    xgb_tuned = XGBClassifier(
        **xgb_search.best_params_,
        eval_metric="logloss",
        random_state=42
    )

    # Retrain on full training set
    xgb_tuned.fit(X_train_tree, y_train)

    # Evaluate on held-out test set
    y_pred_xgb_tuned = xgb_tuned.predict(X_test_tree)
    y_prob_xgb_tuned = xgb_tuned.predict_proba(X_test_tree)[:, 1]

    auc_xgb_tuned = roc_auc_score(y_test, y_prob_xgb_tuned)
    f1_xgb_tuned = f1_score(y_test, y_pred_xgb_tuned)

    # Log best params to MLflow
    mlflow.log_params(xgb_search.best_params_)
    mlflow.log_param("model", "xgboost_tuned")
    mlflow.log_param("n_features", len(SELECTED_FEATURES))

    # Log test set metrics
    mlflow.log_metric("auc", auc_xgb_tuned)
    mlflow.log_metric("f1", f1_xgb_tuned)

    # Log model artifact with input example to fix signature warning
    mlflow.sklearn.log_model(
        xgb_tuned,
        name="xgboost_tuned_model",
        input_example=X_train_tree.iloc[:5]
    )

    print(f"XGBoost Tuned AUC: {auc_xgb_tuned:.4f}")
    print(f"XGBoost Tuned F1:  {f1_xgb_tuned:.4f}")